In [10]:
from cmath import nan

import pandas as pd
import numpy as np


real = pd.read_csv('/Users/macbookpro/platform/Backend/data/historical_data_2025/historical_data_2025Q1/historical_data_time_2025Q1.txt')

In [ ]:
sample.head()

# Restauration des Headers

In [11]:
columns = ['LOAN_SEQUENCE_NUMBER','MONTHLY_REPORTING_PERIOD','CURRENT_ACTUAL_UPB','CURRENT_LOAN_DELINQUENCY_STATUS','LOAN_AGE','REMAINING_MONTHS_TO_LEGAL_MATURITY','DEFECT_SETTLEMENT_DATE','MODIFICATION_FLAG','ZERO_BALANCE_CODE','ZERO_BALANCE_EFFECTIVE_DATE','CURRENT_INTEREST_RATE','CURRENT_NON_INTEREST_BEARING_UPB','DUE_DATE_OF_LAST_PAID_INSTALLMENT','MI_RECOVERIES','NET_SALE_PROCEEDS','NON_MI_RECOVERIES','TOTAL_EXPENSES','LEGAL_COSTS','MAINTENANCE_AND_PRESERVATION_COSTS','TAXES_AND_INSURANCE','MISCELLANEOUS_EXPENSES','ACTUAL_LOSS_CALCULATION','CUMULATIVE_MODIFICATION_COST','INTEREST_RATE_STEP_INDICATOR','PAYMENT_DEFERRAL_FLAG','ESTIMATED_LTV','ZERO_BALANCE_REMOVAL_UPB','DELINQUENT_ACCRUED_INTEREST','DELINQUENCY_DUE_TO_DISASTER','BORROWER_ASSISTANCE_STATUS_CODE','CURRENT_MONTH_MODIFICATION_COST','INTEREST_BEARING_UPB'
]

In [12]:

real = pd.read_csv('/Users/macbookpro/platform/Backend/data/historical_data_2025/historical_data_2025Q1/historical_data_time_2025Q1.txt', header=None, names=columns , sep='|')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_9771/1272140200.py:1: DtypeWarning: Columns (24,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  real = pd.read_csv('/Users/macbookpro/platform/Backend/data/historical_data_2025/historical_data_2025Q1/historical_data_time_2025Q1.txt', header=None, names=columns , sep='|')


In [13]:
real.head()

,LOAN_SEQUENCE_NUMBER,MONTHLY_REPORTING_PERIOD,CURRENT_ACTUAL_UPB,CURRENT_LOAN_DELINQUENCY_STATUS,LOAN_AGE,REMAINING_MONTHS_TO_LEGAL_MATURITY,DEFECT_SETTLEMENT_DATE,MODIFICATION_FLAG,ZERO_BALANCE_CODE,ZERO_BALANCE_EFFECTIVE_DATE,...,CUMULATIVE_MODIFICATION_COST,INTEREST_RATE_STEP_INDICATOR,PAYMENT_DEFERRAL_FLAG,ESTIMATED_LTV,ZERO_BALANCE_REMOVAL_UPB,DELINQUENT_ACCRUED_INTEREST,DELINQUENCY_DUE_TO_DISASTER,BORROWER_ASSISTANCE_STATUS_CODE,CURRENT_MONTH_MODIFICATION_COST,INTEREST_BEARING_UPB
0,F25Q10000004,202502,425000.0,0,0,360,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,57,NaN,NaN,NaN,NaN,NaN,425000.0
1,F25Q10000004,202503,425000.0,0,1,359,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,56,NaN,NaN,NaN,NaN,NaN,425000.0
2,F25Q10000004,202504,425000.0,0,2,358,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,55,NaN,NaN,NaN,NaN,NaN,425000.0
3,F25Q10000004,202505,424000.0,0,3,357,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,55,NaN,NaN,NaN,NaN,NaN,424000.0
4,F25Q10000004,202506,424000.0,0,4,356,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,56,NaN,NaN,NaN,NaN,NaN,424000.0


## Remplacer les valeur de convention par les NaN

- LOAN SEQUENCE NUMBER : identifiant unique du pret, documentation n'assume aucune valeur de convention

- MONTHLY_REPORTING_PERIOD : mois d'observation, documentation n'assume aucune valeur de convention, mais transformation en datetime

In [14]:
real.MONTHLY_REPORTING_PERIOD = pd.to_datetime(real.MONTHLY_REPORTING_PERIOD, format='%Y%m')

- CURRENT ACTUAL UPB : on assume aucune valeur de convention

- CURRENT_LOAN_DELINQUENCY_STATUS : on assume aucune valeur de convention

- LOAN_AGE : on assume aucune valeur de convention

In [15]:
real.LOAN_AGE.unique()

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

- REMAINING_MONTHS_TO_LEGAL_MATURITY : on assume aucune valeur de convention

- DEFECT_SETTLEMENT_DATE : on assume aucune valeur de convention

In [7]:
real.DEFECT_SETTLEMENT_DATE.unique()

array([    nan, 202508., 202506., 202509., 202507., 202510., 202501.,
       202505., 202511., 202504., 202512.])

- MODIFICATION_FLAG : on assume aucune valeur de convention, la colonne est a supprimer
- regle : [Y,P,N]


In [52]:
real['MODIFICATION_FLAG'] = real['MODIFICATION_FLAG'].astype(object)
real.MODIFICATION_FLAG.unique()

array([nan], dtype=object)

- ZERO_BALANCE_CODE :

        01 = Prepaid or Matured (Voluntary Payoff)
        • 02 = Third Party Sale
        • 03 = Short Sale or
         Charge Off
        • 09 = REO Disposition
        • 15 = Whole Loan sales
        • 16 = Reperforming loan
        securitizations
        • 96 = Defect prior to other termination event
    On verifie egalement la coherence avec Les date

In [9]:
real.ZERO_BALANCE_CODE.unique()

array([nan,  1., 96.])

In [55]:

# Cas 1 — Les deux renseignés (normal : prêt clôturé)
cas1 = real[
    real['ZERO_BALANCE_CODE'].notna() &
    real['ZERO_BALANCE_EFFECTIVE_DATE'].notna()
    ].shape[0]

# Cas 2 — Les deux null (normal : prêt actif)
cas2 = real[
    real['ZERO_BALANCE_CODE'].isna() &
    real['ZERO_BALANCE_EFFECTIVE_DATE'].isna()
    ].shape[0]

# Cas 3 — Code renseigné mais date manquante (anomalie à imputer)
cas3 = real[
    real['ZERO_BALANCE_CODE'].notna() &
    real['ZERO_BALANCE_EFFECTIVE_DATE'].isna()
    ].shape[0]

# Cas 4 — Date renseignée mais code manquant (anomalie à signaler)
cas4 = real[
    real['ZERO_BALANCE_CODE'].isna() &
    real['ZERO_BALANCE_EFFECTIVE_DATE'].notna()
    ].shape[0]

print(f"Cas 1 — Les deux renseignés   : {cas1}")
print(f"Cas 2 — Les deux null         : {cas2}")
print(f"Cas 3 — Code seul             : {cas3}  → imputation possible")
print(f"Cas 4 — Date seule            : {cas4}  → anomalie, ne pas imputer")
print(f"\nTotal lignes                  : {real.shape[0]}")
print(f"Contrôle                      : {cas1 + cas2 + cas3 + cas4 == real.shape[0]}")


Cas 1 — Les deux renseignés   : 8016
Cas 2 — Les deux null         : 1419593
Cas 3 — Code seul             : 0  → imputation possible
Cas 4 — Date seule            : 0  → anomalie, ne pas imputer

Total lignes                  : 1427609
Contrôle                      : True


- ZERO_BALANCE_EFFECTIVE_DATE : on assume aucune valeur de convention, ls valeur sont des mois, transformation en datetime

In [18]:
real.ZERO_BALANCE_EFFECTIVE_DATE = pd.to_datetime(real.ZERO_BALANCE_EFFECTIVE_DATE, format='%Y%m')
real.ZERO_BALANCE_EFFECTIVE_DATE.unique()

<DatetimeArray>
[                'NaT', '2025-04-01 00:00:00', '2025-09-01 00:00:00',
 '2025-07-01 00:00:00', '2025-08-01 00:00:00', '2025-06-01 00:00:00',
 '2025-05-01 00:00:00', '2025-03-01 00:00:00', '2025-02-01 00:00:00',
 '2025-10-01 00:00:00', '2025-01-01 00:00:00']
Length: 11, dtype: datetime64[ns]

- CURRENT_INTEREST_RATE : RAS

In [19]:
real.CURRENT_INTEREST_RATE.unique()

array([6.875, 7.125, 6.375, ..., 5.094, 7.804, 8.55 ], shape=(1815,))

- CURRENT_NON_INTEREST_BEARING_UPB : RAS

In [20]:
real.CURRENT_NON_INTEREST_BEARING_UPB.unique()

array([    0.  , 12279.16,  3755.06,  4022.04, 20275.75,  9328.58,
        4583.45,  7812.33])

- DUE_DATE_OF_LAST_PAID_INSTALLMENT : RAS, transformation en datetime

In [21]:
real.DUE_DATE_OF_LAST_PAID_INSTALLMENT = pd.to_datetime(real.DUE_DATE_OF_LAST_PAID_INSTALLMENT, format='%Y%m')
real.DUE_DATE_OF_LAST_PAID_INSTALLMENT.unique()

<DatetimeArray>
[                'NaT', '2025-04-01 00:00:00', '2025-09-01 00:00:00',
 '2025-07-01 00:00:00', '2025-06-01 00:00:00', '2025-05-01 00:00:00',
 '2025-08-01 00:00:00', '2025-02-01 00:00:00', '2025-03-01 00:00:00',
 '2055-02-01 00:00:00', '2025-10-01 00:00:00', '2025-11-01 00:00:00',
 '2026-01-01 00:00:00', '2055-03-01 00:00:00', '2025-12-01 00:00:00']
Length: 15, dtype: datetime64[ns]

- MI_RECOVERIES : Que des Nan, #todo trouver son mode d'obtention

In [22]:
real.MI_RECOVERIES.unique()

array([nan])

- NET_SALE_PROCEEDS : Que des Nan, #todo trouver son mode d'obtention

In [23]:
real.NET_SALE_PROCEEDS.isna().mean()

np.float64(1.0)

In [24]:
real.NET_SALE_PROCEEDS.unique()

array([nan])

- NON_MI_RECOVERIE : que des Nan, #todo trouver son mode d'obtention

In [26]:
real.NON_MI_RECOVERIES.unique()

array([nan])

- TOTAL_EXPENSES : Que des NAN

In [27]:
real.TOTAL_EXPENSES.unique()

array([nan])

- Legal Costs : Que des Nan

In [28]:
real.LEGAL_COSTS.unique()

array([nan])

- Maintenance and Preservation Costs : Que des Nan

In [29]:
real.MAINTENANCE_AND_PRESERVATION_COSTS.unique()

array([nan])

- Tax and Insurance : Que des Nan

In [30]:
real.TAXES_AND_INSURANCE.unique()

array([nan])

- Miscellaneous Expenses : Que des Nan

In [31]:
real.MISCELLANEOUS_EXPENSES.unique()

array([nan])

- Actual Loss Calculation : Que des Nan

In [32]:
real.ACTUAL_LOSS_CALCULATION.unique()

array([nan])

- Cumulative Modification Cost : aucune convention signaler

In [33]:
real.CUMULATIVE_MODIFICATION_COST.unique()

array([   nan, 141.22,  20.81,  42.06, 202.76, 302.22,  27.79,  27.02])

- Interest Rate Step Indicator : aucune convention signaler

In [34]:
real.INTEREST_RATE_STEP_INDICATOR.unique()

array([nan])

- Payment deferment flag : aucune convention signaler

In [35]:
real.PAYMENT_DEFERRAL_FLAG.unique()

array([nan, 'Y', 'P'], dtype=object)

- Estimated LTV: convention detecter

In [36]:
real.ESTIMATED_LTV.isna().mean()

np.float64(0.0)

In [37]:
real['ESTIMATED_LTV'] = real['ESTIMATED_LTV'].replace(999, np.nan)

In [38]:
real.ESTIMATED_LTV.isna().mean()

np.float64(0.11501678680927341)

- Zero Balance Removal UPB : aucune convention signaler



In [ ]:
real.ZERO_BALANCE_REMOVAL_UPB.isna().mean()

- Delinquent Accrued Interest : aucune convention signaler


In [39]:
real.DELINQUENT_ACCRUED_INTEREST.unique()

array([nan])

- Borrower BORROWER ASSISTANCE STATUS CODE

In [40]:
real.BORROWER_ASSISTANCE_STATUS_CODE.unique()

array([nan, 'F', 'R', 'T'], dtype=object)

- CURRENT MONTH MODIFICATION COST -

In [41]:
real.CURRENT_MONTH_MODIFICATION_COST.unique()

array([   nan,  70.61,  20.81,  21.03, 101.38,  50.37,  27.79,  27.02])

- INTEREST BEARING UPB

In [42]:
real.INTEREST_BEARING_UPB.unique()

array([425000.  , 424000.  , 423000.  , ..., 762073.91, 326866.01,
       120506.75], shape=(52538,))

# Cleaning

-       Suppresion des variable Post-decision
-       les variable a 100 % de NaN seront supprimer si il s'agit de variable Post-decision
-       Imputation des variables restantes par mediane mode

In [43]:
real.isna().mean()

LOAN_SEQUENCE_NUMBER                  0.000000
MONTHLY_REPORTING_PERIOD              0.000000
CURRENT_ACTUAL_UPB                    0.000000
CURRENT_LOAN_DELINQUENCY_STATUS       0.000000
LOAN_AGE                              0.000000
REMAINING_MONTHS_TO_LEGAL_MATURITY    0.000000
DEFECT_SETTLEMENT_DATE                0.999802
MODIFICATION_FLAG                     1.000000
ZERO_BALANCE_CODE                     0.994385
ZERO_BALANCE_EFFECTIVE_DATE           0.994385
CURRENT_INTEREST_RATE                 0.000000
CURRENT_NON_INTEREST_BEARING_UPB      0.000000
DUE_DATE_OF_LAST_PAID_INSTALLMENT     0.994385
MI_RECOVERIES                         1.000000
NET_SALE_PROCEEDS                     1.000000
NON_MI_RECOVERIES                     1.000000
TOTAL_EXPENSES                        1.000000
LEGAL_COSTS                           1.000000
MAINTENANCE_AND_PRESERVATION_COSTS    1.000000
TAXES_AND_INSURANCE                   1.000000
MISCELLANEOUS_EXPENSES                1.000000
ACTUAL_LOSS_C

In [44]:
ELIGIBLE_COLUMNS = [
    "LOAN_SEQUENCE_NUMBER",
    "MONTHLY_REPORTING_PERIOD",
    "CURRENT_ACTUAL_UPB",
    "CURRENT_LOAN_DELINQUENCY_STATUS",
    "LOAN_AGE",
    "REMAINING_MONTHS_TO_LEGAL_MATURITY",
    "MODIFICATION_FLAG",
    "ZERO_BALANCE_CODE",
    "ZERO_BALANCE_EFFECTIVE_DATE",
    "CURRENT_INTEREST_RATE",
    "CURRENT_NON_INTEREST_BEARING_UPB",
    "DUE_DATE_OF_LAST_PAID_INSTALLMENT",
    "INTEREST_RATE_STEP_INDICATOR",
    "ESTIMATED_LTV",
    "DELINQUENCY_DUE_TO_DISASTER",
    "BORROWER_ASSISTANCE_STATUS_CODE",
    "INTEREST_BEARING_UPB",
]

In [49]:
to_drop = [col for col in real.columns if real[col].isna().mean() >= 0.9  and col not in ELIGIBLE_COLUMNS]
to_drop

[]

In [50]:
real = real.drop(to_drop, axis=1)

In [51]:
real.isna().mean()

LOAN_SEQUENCE_NUMBER                  0.000000
MONTHLY_REPORTING_PERIOD              0.000000
CURRENT_ACTUAL_UPB                    0.000000
CURRENT_LOAN_DELINQUENCY_STATUS       0.000000
LOAN_AGE                              0.000000
REMAINING_MONTHS_TO_LEGAL_MATURITY    0.000000
MODIFICATION_FLAG                     1.000000
ZERO_BALANCE_CODE                     0.994385
ZERO_BALANCE_EFFECTIVE_DATE           0.994385
CURRENT_INTEREST_RATE                 0.000000
CURRENT_NON_INTEREST_BEARING_UPB      0.000000
DUE_DATE_OF_LAST_PAID_INSTALLMENT     0.994385
INTEREST_RATE_STEP_INDICATOR          1.000000
ESTIMATED_LTV                         0.115017
DELINQUENCY_DUE_TO_DISASTER           0.999962
BORROWER_ASSISTANCE_STATUS_CODE       0.999755
INTEREST_BEARING_UPB                  0.000000
dtype: float64

In [56]:
real.ZERO_BALANCE_CODE.unique()

array([nan,  1., 96.])

In [57]:
real[real['ZERO_BALANCE_CODE'] == 96]['LOAN_SEQUENCE_NUMBER'].nunique()

272

Décision pour la modélisation :

        Ces 272 prêts sont à exclure de la population de modélisation pour deux raisons :

        La terminaison n'est pas liée au comportement du client mais à un problème administratif de souscription
        Ils introduisent un biais — leur sortie du pool est exogène au risque de crédit

        Les NaN ici ne sont pas des valeurs manquantes à imputer — ce sont des prêts actifs.
        Un prêt actif n'a pas de code de clôture ni de date de clôture par définition. Ces NaN sont corrects et attendus.

        Les date lier au pret actif sont garder en nat

Remplacer les NAN PAR 0. comme un code

    Le Taux de NAN des DUE_DATE_OF_LAST_PAID_INSTALLMENT  est proportionnel au Pret actif ,
    DUE_DATE_OF_LAST_PAID_INSTALLMENT n'est renseignée que quand le client a un retard de paiement.
     Sur un prêt actif et à jour, cette date n'existe pas encore. Aucun client dans 2025Q1 n'est en retard

ON Garde DUE_DATE_OF_LAST_PAID_INSTALLMENT en Nat

In [58]:
real = real[real['ZERO_BALANCE_CODE'] != 96]

# Vérification
print(f"Lignes restantes     : {real.shape[0]}")
print(f"Prêts uniques        : {real['LOAN_SEQUENCE_NUMBER'].nunique()}")
print(f"ZERO_BALANCE_CODE    : {real['ZERO_BALANCE_CODE'].unique()}")

Lignes restantes     : 1427337
Prêts uniques        : 210748
ZERO_BALANCE_CODE    : [nan  1.]


In [62]:
real['ZERO_BALANCE_CODE'] = real['ZERO_BALANCE_CODE'].fillna(0)

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_9771/582219271.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real['ZERO_BALANCE_CODE'] = real['ZERO_BALANCE_CODE'].fillna(0)


LTV

                ESTIMATED_LTV — 11.5% de NaN concentrés sur les prêts jeunes (LOAN_AGE ≤ 8 mois, médiane 1 mois). Freddie Mac ne calcule pas l'AVM sur les
                premiers mois de vie du prêt, la valeur d'origination étant encore représentative. Stratégie d'imputation post-jointure : combler les NaN par
                 le LTV d'origination issu du fichier origination via LOAN_SEQUENCE_NUMBER.

                LTV d'origination est dans le fichier origination — donc imputation possible uniquement après la jointure via LOAN_SEQUENCE_NUMBER.

In [66]:
# Est-ce que les NaN correspondent à des prêts récents (LOAN_AGE faible) ?
print(real[real['ESTIMATED_LTV'].isna()]['LOAN_AGE'].describe())

# Ou à un type de bien particulier ?
print(real[real['ESTIMATED_LTV'].isna()]['LOAN_AGE'].value_counts().head(10))

count    164056.000000
mean          1.810120
std           2.107498
min           0.000000
25%           0.000000
50%           1.000000
75%           3.000000
max           8.000000
Name: LOAN_AGE, dtype: float64
LOAN_AGE
0    70104
1    25955
2    15319
5    13433
3    13395
4    13160
6     8496
7     4150
8       44
Name: count, dtype: int64


delinquency- due deasastre

In [67]:
real['DELINQUENCY_DUE_TO_DISASTER'] = real['DELINQUENCY_DUE_TO_DISASTER'].fillna('N')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_9771/1311644339.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real['DELINQUENCY_DUE_TO_DISASTER'] = real['DELINQUENCY_DUE_TO_DISASTER'].fillna('N')


INTEREST RATE STEP INDICATOR

In [68]:
real['INTEREST_RATE_STEP_INDICATOR'] = real['INTEREST_RATE_STEP_INDICATOR'].fillna('N')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_9771/399061449.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real['INTEREST_RATE_STEP_INDICATOR'] = real['INTEREST_RATE_STEP_INDICATOR'].fillna('N')


BORROWER_ASSISTANCE_STATUS_CODE

In [69]:
real['BORROWER_ASSISTANCE_STATUS_CODE'] = real['BORROWER_ASSISTANCE_STATUS_CODE'].fillna('N')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_9771/2991392960.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real['BORROWER_ASSISTANCE_STATUS_CODE'] = real['BORROWER_ASSISTANCE_STATUS_CODE'].fillna('N')


Documentation :


            Périmètre dataset 2025 (Q1, Q2, Q3) — Le dataset 2025 ne contient aucun événement de défaut (ZERO_BALANCE_CODE ∈ {02, 03, 09}),
             ni de délinquance liée à une catastrophe naturelle significative. Il est utilisable pour valider la pipeline de traitement et scorer les prêts
              actifs. Pour la modélisation PD/LGD, les millésimes 2007-2010 (crise subprimes) et 2020-2021 (COVID) devront être téléchargés depuis Clarity Data
               Intelligence — ils constituent les principales sources d'événements de défaut et de forbearance dans l'historique Freddie Mac.